# 数值输入不标准化会怎样？

对比纯高斯 KAN 和 MLP。每个模型分别跑「标准化」与「原始数值」两种输入，并在随机按行和按建筑形状两套划分上评估。将本 Notebook 与项目的 ENB2012_data.xlsx 放在同一目录后，可从头运行。

仅改变数值输入的标准化。两种输入都保留朝向、玻璃分布独热编码，也都按训练集统计量标准化供暖与制冷目标。预测值还原到原始尺度后再计算 MAE 和 RMSE。每种条件均独立在验证集上选配置和早停。KAN 高斯中心固定为 −2.5 到 2.5，尺度为 0.5。

In [1]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"

import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd



DATA = Path("ENB2012_data.xlsx")
FEATURES = ["X" + str(i) for i in range(1, 9)]
TARGETS = ["Y1", "Y2"]
NAMES = ["Heating", "Cooling"]
NUMERIC = ["X1", "X2", "X3", "X4", "X5", "X7"]
CATEGORIES = {"X6": (2, 3, 4, 5), "X8": (0, 1, 2, 3, 4, 5)}
SPLIT_SEED = 2026
TUNE_SEED = 17
FINAL_SEEDS = (101, 202, 303)
MAX_EPOCHS = 250
PATIENCE = 35
BATCH_SIZE = 64


## 数据与划分

每套划分只用训练集估计均值和标准差。划分索引与主实验一致。

In [2]:
df = pd.read_excel(DATA, usecols=FEATURES + TARGETS)
assert df.shape == (768, 10)
assert df.notna().all().all() and not df[FEATURES].duplicated().any()
assert np.isfinite(df.to_numpy()).all()
for key, values in CATEGORIES.items():
    assert set(df[key]) == set(values)

# 先按固定类别展开 X6 和 X8，再与六个数值特征拼接。
input_columns = NUMERIC + [
    f"{field}_{level}"
    for field, levels in CATEGORIES.items()
    for level in levels
]
category_indicators = [
    (df[field].to_numpy()[:, None] == levels).astype(np.float32)
    for field, levels in CATEGORIES.items()
]
encoded_inputs = pd.DataFrame(
    np.column_stack([df[NUMERIC].to_numpy(np.float32), *category_indicators]),
    columns=input_columns,
)
assert encoded_inputs.shape == (768, 16)
print(f"编码后：{encoded_inputs.shape[0]} 行 × {encoded_inputs.shape[1]} 列")
print(encoded_inputs.head(4).to_string(index=False))

def preprocess(split):
    train = df.iloc[split["train"]]
    def scale(cols):
        mean = train[cols].mean().to_numpy(np.float32)
        sd = train[cols].std(ddof=0).to_numpy(np.float32)
        sd = np.where(sd > 0, sd, 1)
        return ((df[cols].to_numpy(np.float32) - mean) / sd, mean, sd)
    design = encoded_inputs.to_numpy(copy=True)
    scaled = design.copy()
    scaled[:, :len(NUMERIC)], _, _ = scale(NUMERIC)
    target, y_mean, y_std = scale(TARGETS)
    return {"raw": design, "standardized": scaled}, target, y_mean, y_std

SPLITS = {}
DESIGN = {}
manifest = pd.DataFrame({"row_id": np.arange(len(df))})

def register_split(name, split):
    indices = np.sort(np.concatenate(list(split.values())))
    assert np.array_equal(indices, np.arange(len(df)))
    for part, idx in split.items():
        manifest.loc[idx, name] = part
    design = preprocess(split)
    assert all(x.shape == (768, 16) for x in design[0].values())
    SPLITS[name] = split
    DESIGN[name] = design


row = np.random.default_rng(SPLIT_SEED).permutation(len(df))
register_split("row", dict(train=row[:538], val=row[538:653], test=row[653:]))

geometry_keys = sorted(set(df[FEATURES[:5]].itertuples(index=False, name=None)))
geometry_map = {g: i + 1 for i, g in enumerate(geometry_keys)}
geometry_id = np.asarray([geometry_map[g] for g in
                          df[FEATURES[:5]].itertuples(index=False, name=None)])
assert len(geometry_keys) == 12
shapes = np.random.default_rng(SPLIT_SEED).permutation(np.arange(1, 13))
shape_split = dict(
    train=np.flatnonzero(np.isin(geometry_id, np.r_[shapes[:7], shapes[9]])),
    val=np.flatnonzero(np.isin(geometry_id, shapes[7:9])),
    test=np.flatnonzero(np.isin(geometry_id, shapes[10:])),
)
manifest["geometry_id"] = geometry_id
manifest = manifest[["row_id", "geometry_id", "row"]]
register_split("shape", shape_split)
assert all(manifest.groupby("geometry_id")["shape"].nunique() == 1)
for part, group in manifest.groupby("shape", sort=False):
    ids = sorted(group["geometry_id"].unique().tolist())
    print(f"{part}: {len(group)} rows, geometry IDs {ids}")



编码后：768 行 × 16 列
  X1    X2    X3     X4  X5  X7  X6_2  X6_3  X6_4  X6_5  X8_0  X8_1  X8_2  X8_3  X8_4  X8_5
0.98 514.5 294.0 110.25 7.0 0.0   1.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0
0.98 514.5 294.0 110.25 7.0 0.0   0.0   1.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0
0.98 514.5 294.0 110.25 7.0 0.0   0.0   0.0   1.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0
0.98 514.5 294.0 110.25 7.0 0.0   0.0   0.0   0.0   1.0   1.0   0.0   0.0   0.0   0.0   0.0
train: 512 rows, geometry IDs [1, 2, 3, 7, 8, 9, 10, 12]
test: 128 rows, geometry IDs [5, 11]
val: 128 rows, geometry IDs [4, 6]


## 两个模型与训练规则

复用主实验的纯高斯 KAN 和 MLP。每个模型尝试原来的 6 个候选配置，使用种子 17 调参，再用 101、202、303 分别训练和测试。

In [3]:
CENTERS = np.linspace(-2.5, 2.5, 11, dtype=np.float32)
GAUSSIAN_SCALE = np.float32(0.5)

def gaussian_basis(x):
    u = (x[:, :, None] - CENTERS) / GAUSSIAN_SCALE
    basis = np.exp(-u**2)
    derivative = (-2 * u / GAUSSIAN_SCALE) * basis
    return basis, derivative

class GaussianLayer:
    def __init__(self, inputs, outputs, rng):
        factors = len(CENTERS)
        self.weights = rng.normal(0, 1 / np.sqrt(inputs * factors),
                                  (inputs * factors, outputs)).astype(np.float32)
        self.bias = np.zeros(outputs, np.float32)

    def forward(self, x):
        basis, derivative = gaussian_basis(x)
        self.x = basis.reshape(len(x), -1)
        self.derivative = derivative
        return self.x @ self.weights + self.bias

    def backward(self, output_gradient):
        dw = self.x.T @ output_gradient
        db = output_gradient.sum(axis=0)
        incoming = output_gradient @ self.weights.T
        incoming = (incoming.reshape(self.derivative.shape) * self.derivative).sum(axis=2)
        return incoming, (dw, db)

class GaussianKAN:
    def __init__(self, width, seed, inputs=16):
        rng = np.random.default_rng(seed)
        self.first = GaussianLayer(inputs, width, rng)
        self.second = GaussianLayer(width, 2, rng)
        self.parameters = [self.first.weights, self.first.bias,
                           self.second.weights, self.second.bias]

    def predict(self, x):
        hidden = self.first.forward(x)
        return self.second.forward(hidden)

    def backward(self, grad):
        delta, second = self.second.backward(grad)
        _, first = self.first.backward(delta)
        return (*first, *second)

def kan_gradient_check():
    rng = np.random.default_rng(4)
    x = rng.normal(size=(4, 16)).astype(np.float32)
    y = rng.normal(size=(4, 2)).astype(np.float32)
    net = GaussianKAN(3, 4)
    residual = net.predict(x) - y
    grads = net.backward(2 * residual / residual.size)
    for param, grad in zip(net.parameters, grads, strict=True):
        idx = np.unravel_index(np.argmax(np.abs(grad)), grad.shape)
        value = param[idx].copy()
        eps = 0.001
        param[idx] = value + eps
        right = ((net.predict(x) - y)**2).mean()
        param[idx] = value - eps
        left = ((net.predict(x) - y)**2).mean()
        param[idx] = value
        np.testing.assert_allclose(grad[idx], (right-left)/(2*eps), rtol=0.02, atol=0.0002)

kan_gradient_check()
# 中心处高斯基函数为 1，斜率为 0。离中心越远，基函数值越小。
probe = np.array([[0.0], [0.5], [1.0], [-0.5]], dtype=np.float64)
values, derivatives = gaussian_basis(probe)
middle = len(CENTERS) // 2
np.testing.assert_allclose(values[:, 0, middle], np.exp(-np.array([0, 1, 4, 1])))
eps = 1e-5
left, _ = gaussian_basis(probe - eps)
right, _ = gaussian_basis(probe + eps)
np.testing.assert_allclose(derivatives, (right - left) / (2 * eps), atol=1e-8)
print("高斯基函数与网络的梯度检查通过。")

def silu(x):
    sigmoid = 1 / (1 + np.exp(-np.clip(x, -60, 60)))
    return x * sigmoid, sigmoid * (1 + x * (1 - sigmoid))

class DenseLayer:
    def __init__(self, inputs, outputs, rng):
        self.weights = rng.normal(0, 1 / np.sqrt(inputs),
                                  (inputs, outputs)).astype(np.float32)
        self.bias = np.zeros(outputs, np.float32)

    def forward(self, x):
        self.x = x
        return self.x @ self.weights + self.bias

    def backward(self, output_gradient):
        dw = self.x.T @ output_gradient
        db = output_gradient.sum(axis=0)
        incoming = output_gradient @ self.weights.T
        return incoming, (dw, db)

class MLP:
    def __init__(self, width, seed, inputs=16):
        rng = np.random.default_rng(seed)
        self.first = DenseLayer(inputs, width, rng)
        self.second = DenseLayer(width, 2, rng)
        self.parameters = [self.first.weights, self.first.bias,
                           self.second.weights, self.second.bias]

    def predict(self, x):
        hidden = self.first.forward(x)
        hidden, self.hidden_derivative = silu(hidden)
        return self.second.forward(hidden)

    def backward(self, grad):
        delta, second = self.second.backward(grad)
        delta = delta * self.hidden_derivative
        _, first = self.first.backward(delta)
        return (*first, *second)

def mlp_gradient_check():
    rng = np.random.default_rng(4)
    x = rng.normal(size=(4, 16)).astype(np.float32)
    y = rng.normal(size=(4, 2)).astype(np.float32)
    net = MLP(3, 4)
    residual = net.predict(x) - y
    grads = net.backward(2 * residual / residual.size)
    for param, grad in zip(net.parameters, grads, strict=True):
        idx = np.unravel_index(np.argmax(np.abs(grad)), grad.shape)
        value = param[idx].copy()
        eps = 0.001
        param[idx] = value + eps
        right = ((net.predict(x) - y)**2).mean()
        param[idx] = value - eps
        left = ((net.predict(x) - y)**2).mean()
        param[idx] = value
        np.testing.assert_allclose(grad[idx], (right-left)/(2*eps), rtol=0.02, atol=0.0002)

mlp_gradient_check()
print("MLP 梯度检查通过。")

高斯基函数与网络的梯度检查通过。
MLP 梯度检查通过。


In [4]:
@dataclass(frozen=True)
class Config:
    model: str
    width: int
    lr: float

MODELS = {"KAN": GaussianKAN, "MLP": MLP}
WIDTHS = {"KAN": (4, 6, 8), "MLP": (47, 71, 95)}
LEARNING_RATES = (0.003, 0.01)

def fit(x, y, train, val, cfg, seed):
    net = MODELS[cfg.model](cfg.width, seed)
    rng = np.random.default_rng(seed + 10000)
    params = net.parameters
    first = [np.zeros_like(p) for p in params]
    second = [np.zeros_like(p) for p in params]
    best, best_epoch, step = float("inf"), 0, 0
    history = []
    start = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        shuffled = rng.permutation(train)
        for offset in range(0, len(shuffled), BATCH_SIZE):
            idx = shuffled[offset:offset+BATCH_SIZE]
            err = net.predict(x[idx]) - y[idx]
            grads = net.backward(2 * err / err.size)
            step += 1
            alpha = cfg.lr * np.sqrt(1 - 0.999**step) / (1 - 0.9**step)
            epsilon = 1e-8 * np.sqrt(1 - 0.999**step)
            for p, g, m, v in zip(params, grads, first, second, strict=True):
                m *= 0.9
                m += 0.1 * g
                v *= 0.999
                v += 0.001 * g**2
                p -= alpha * m / (np.sqrt(v) + epsilon)
        score = float(np.mean((net.predict(x[val]) - y[val])**2))
        if not np.isfinite(score):
            raise FloatingPointError(f"Nonfinite validation loss for {cfg}")
        history.append(score)
        if score < best:
            best, best_epoch = score, epoch
            snapshot = [p.copy() for p in params]
        if epoch - best_epoch >= PATIENCE:
            break
    for p, saved in zip(params, snapshot, strict=True):
        p[...] = saved
    seconds = time.perf_counter() - start
    return net, dict(val_mse=best, best_epoch=best_epoch, epochs=len(history),
                     seconds=seconds, params=sum(p.size for p in params)), history



## 配对实验和测试结果

在每一种模型、划分和输入条件下单独选择验证误差最低的配置。下表报告三次测试的均值。

In [5]:
runs = []
for protocol, split in SPLITS.items():
    designs, y, y_mean, y_std = DESIGN[protocol]
    test = split["test"]
    actual = df.iloc[test][TARGETS].to_numpy()
    for mode, x in designs.items():
        for model, widths in WIDTHS.items():
            ranked = []
            for width in widths:
                for lr in LEARNING_RATES:
                    cfg = Config(model, width, lr)
                    _, stat, _ = fit(x, y, split["train"], split["val"], cfg, TUNE_SEED)
                    ranked.append((stat["val_mse"], cfg))
            _, cfg = min(ranked, key=lambda trial: trial[0])
            for seed in FINAL_SEEDS:
                net, stats, _ = fit(x, y, split["train"], split["val"], cfg, seed)
                prediction = net.predict(x[test]) * y_std + y_mean
                assert np.isfinite(prediction).all()
                residual = prediction - actual
                result = dict(protocol=protocol, model=model, inputs=mode, seed=seed,
                              width=cfg.width, lr=cfg.lr, **stats)
                for j, load in enumerate(NAMES):
                    result[load + "_MAE"] = np.abs(residual[:, j]).mean()
                    result[load + "_RMSE"] = np.sqrt(np.square(residual[:, j]).mean())
                runs.append(result)

runs = pd.DataFrame(runs)
assert len(runs) == 24
metrics = ["Heating_MAE", "Cooling_MAE", "Heating_RMSE", "Cooling_RMSE", "seconds", "params"]
grouped = runs.groupby(["protocol", "model", "inputs"])[metrics]
summary = grouped.mean()
print(summary.round(4).to_string())
print("\n原始数值 MAE 减去标准化 MAE（正数表示原始数值的误差更大）")
mae = summary[["Heating_MAE", "Cooling_MAE"]]
print((mae.xs("raw", level="inputs") - mae.xs("standardized", level="inputs")).round(4).to_string())


                             Heating_MAE  Cooling_MAE  Heating_RMSE  Cooling_RMSE  seconds  params
protocol model inputs                                                                             
row      KAN   raw                0.9095       1.4291        1.0949        1.7348   0.3531  1196.0
               standardized       0.4184       0.7007        0.5569        1.0314   0.1854  1196.0
         MLP   raw                2.4064       2.7159        3.2403        3.7096   0.1693   895.0
               standardized       0.3628       0.8360        0.4679        1.2098   0.1807  1351.0
shape    KAN   raw                7.9956       7.8524        8.5644        8.3467   0.0433   798.0
               standardized       4.7344       4.3531        5.1003        5.1601   0.0446  1196.0
         MLP   raw               11.3299      49.5303       13.6437       66.0273   0.0724   895.0
               standardized       1.2653       1.5324        1.5834        2.0284   0.0270  1351.0

原始数值 MAE 

## 为什么固定网格会影响 KAN？

统计每个数值特征的 11 个高斯基函数是否在第一层全部变成浮点零。用同一批测试行比较两套输入。

In [6]:
rows = []
for protocol, split in SPLITS.items():
    for mode, x in DESIGN[protocol][0].items():
        numeric = x[split["test"], :len(NUMERIC)]
        basis, _ = gaussian_basis(numeric)
        inactive = np.all(basis == 0, axis=2).mean(axis=0)
        rows.append(dict(protocol=protocol, inputs=mode,
                         **dict(zip(NUMERIC, inactive, strict=True))))
print(pd.DataFrame(rows).set_index(["protocol", "inputs"]).to_string())


                        X1   X2   X3   X4   X5   X7
protocol inputs                                    
row      raw           0.0  1.0  1.0  1.0  0.0  0.0
         standardized  0.0  0.0  0.0  0.0  0.0  0.0
shape    raw           0.0  1.0  1.0  1.0  0.0  0.0
         standardized  0.0  0.0  0.0  0.0  0.0  0.0


## 本次运行的结果

| 划分 | 模型 | 供暖 MAE（标准化 → 原始） | 制冷 MAE（标准化 → 原始） |
|---|---|---:|---:|
| 随机按行 | KAN | 0.418 → 0.909 | 0.701 → 1.429 |
| 随机按行 | MLP | 0.363 → 2.406 | 0.836 → 2.716 |
| 按建筑形状 | KAN | 4.734 → 7.996 | 4.353 → 7.852 |
| 按建筑形状 | MLP | 1.265 → 11.330 | 1.532 → 49.530 |

在现有固定网格下，直接输入原始面积时，面积对应的 11 个高斯值全部变成浮点零。因此这些面积列经第一层变换后会失去区分样本的能力。这项实验评估的是原有架构和搜索范围下取消输入标准化的效果。